<div class="alert alert-block alert-info" style="padding:20px;color:white;margin:auto;font-size:300%;text-align:center;display:fill;border-radius:60px;background-color:#37006f;overflow:hidden;font-weight:800">Text to Midi</div>

This notebooks tackle an approach for generating a MIDI file based on caption provided to the model.

## 1. Setup

In [1]:
cd ..

d:\Personal Projects\tune-ml


d:\Personal Projects\tune-ml\.venv\lib\site-packages\IPython\core\magics\osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
# imports
from tuneml.data.midi import MidiCapDataset

d:\Personal Projects\tune-ml\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Data Processing

In [3]:
dataset = MidiCapDataset("datasets/midicaps/train.json", n_samples=100)
print(f"Dataset size: {len(dataset)}")
print(f"Example MIDI shape: {dataset.midi[0].shape}")

KeyboardInterrupt: 

## 3. Model Training

In [5]:
from tuneml.tokenizers.MidiTokenizer import MidiTokenizer
from tuneml.tokenizers.TextTokenizer import Flan5Tokenizer

midi_tokenizer = MidiTokenizer()
text_tokenizer = Flan5Tokenizer()

# Hyperparameters
datapath = "datasets/midicaps/train.pt"
batch_size = 1
warmup_steps = 2000
ckpt_path = "weights/midi_transformer_ckpt.pt"
load_checkpoint = False
save_path = "weights/midi_transformer_model.pt"
num_epochs = 10
hparams = {
    "d_model": 128,
    "num_layers": 2,
    "num_heads": 8,
    "d_ff": 512,
    "max_abs_position": 2048,
    "midi_vocab_size": midi_tokenizer.vocab_size,
    "text_vocab_size": text_tokenizer.vocab_size,
    "bias": True,
    "dropout": 0.1,
    "layernorm_eps": 1e-6,
}

In [5]:
# # set up the trainer
# print("Setting up the trainer...")
# trainer = MidiTransformerTrainer(
#     hparams,
#     datapath,
#     batch_size,
#     warmup_steps,
#     ckpt_path,
#     load_checkpoint,
#     n_samples=100,
#  )
# print()

# # train the model
# trainer.fit(num_epochs)

# # done training, save the model
# print("Saving...")
# save_file = {
#     "state_dict": trainer.model.state_dict(),
#     "hparams": trainer.hparams
# }
# torch.save(save_file, save_path)
# print("Done!")

## 4. Model Evaluation

In [4]:
from tuneml.modules.generator.MidiGenerator import MidiGenerator

In [6]:
# set up the generator
generator = MidiGenerator(model_ckpt_path=save_path)

CUDA is not available, falling back to CPU


In [ ]:
# generate a MIDI from text
midi_tokens = generator("A happy, upbeat melody with a catchy rhythm and bright instrumentation.")

CUDA is not available, falling back to CPU


d:\Personal Projects\tune-ml\.venv\lib\site-packages\torch\nn\modules\transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\NestedTensorImpl.cpp:182.)
  output = torch._nested_tensor_from_mask(


In [ ]:
# convert the generated MIDI tokens back to a MIDI file
midi_obj = generator.tokens_to_midi(
    tokens=midi_tokens,
    fname="generated_melody",
    save_path="./"
)

In [ ]:
# play the generated MIDI (requires pygame)
generator.play_midi(
    "./datasets/midicaps/lmd_full/0/0a0a2b0e4d3b7bf4c5383ba025c4683e.mid",
    soundfont_path="./soundfonts/The Ultimate SoundFont Pack/Ultimate Guitar Kit 2.SF2",
)

In [6]:
import mido
midi_obj = mido.MidiFile("./datasets/midicaps/lmd_full/0/0a0a2b0e4d3b7bf4c5383ba025c4683e.mid")
midi_obj

MidiFile(type=1, ticks_per_beat=480, tracks=[
  MidiTrack([
    MetaMessage('copyright', text='Copyright © 1996 by NoteWorthy ArtWare, Inc.\rAll Rights Reserved', time=0),
    MetaMessage('time_signature', numerator=4, denominator=4, clocks_per_click=24, notated_32nd_notes_per_beat=8, time=0),
    MetaMessage('set_tempo', tempo=500000, time=0),
    MetaMessage('key_signature', key='E', time=0),
    MetaMessage('end_of_track', time=0)]),
  MidiTrack([
    MetaMessage('track_name', name='Acoustic Grand Piano', time=0),
    Message('program_change', channel=0, program=0, time=0),
    MetaMessage('lyrics', text='Generated by NoteWorthy Composer', time=0),
    MetaMessage('end_of_track', time=0)]),
  MidiTrack([
    MetaMessage('track_name', name='Drums2', time=0),
    Message('program_change', channel=9, program=4, time=0),
    MetaMessage('midi_port', port=0, time=0),
    Message('program_change', channel=9, program=4, time=0),
    Message('control_change', channel=9, control=7, value=127

In [2]:
from miditok import REMI, TokenizerConfig

# Our parameters
TOKENIZER_PARAMS = {
    "pitch_range": (21, 109),
    "beat_res": {(0, 4): 8, (4, 12): 4},
    "num_velocities": 32,
    "special_tokens": ["PAD", "BOS", "EOS", "MASK"],
    "use_chords": True,
    "use_rests": False,
    "use_tempos": True,
    "use_time_signatures": False,
    "use_programs": True,
    "num_tempos": 32,  # number of tempo bins
    "tempo_range": (40, 250),  # (min, max)
}
config = TokenizerConfig(**TOKENIZER_PARAMS)

# Creates the tokenizer
tokenizer = REMI(config)

d:\Personal Projects\tune-ml\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\Personal Projects\tune-ml\.venv\lib\site-packages\miditok\tokenizations\remi.py:88: UserWarning: Attribute controls are not compatible with 'config.one_token_stream_for_programs' and multi-vocabulary tokenizers. Disabling them from the config.
  super().__init__(tokenizer_config, params)


In [3]:
from pathlib import Path

# Tokenize a MIDI file
tokens = tokenizer(Path("../datasets/midicaps/lmd_full/0/0a0a2b0e4d3b7bf4c5383ba025c4683e.mid"))

# Convert to MIDI and save it
generated_midi = tokenizer(tokens)
generated_midi.dump_midi("./generated_melody.mid")

In [7]:
dir(tokens)

['_TokSequence__slice',
 '__add__',
 '__annotations__',
 '__class__',
 '__dataclass_fields__',
 '__dataclass_params__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getitem__',
 '__gt__',
 '__hash__',
 '__iadd__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__len__',
 '__lt__',
 '__match_args__',
 '__module__',
 '__ne__',
 '__new__',
 '__radd__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_ids_decoded',
 '_split_per_ticks',
 '_ticks_bars',
 '_ticks_beats',
 'are_ids_encoded',
 'bytes',
 'events',
 'ids',
 'split_per_bars',
 'split_per_beats',
 'tokens']

In [ ]:
# play the generated MIDI (requires pygame)
generator.play_midi(
    "./notebooks/generated_melody.mid",
    soundfont_path="./soundfonts/The Ultimate SoundFont Pack/Ultimate Guitar Kit 2.SF2",
)

In [4]:
tokenizer.vocab

{'PAD_None': 0,
 'BOS_None': 1,
 'EOS_None': 2,
 'MASK_None': 3,
 'Bar_None': 4,
 'Pitch_21': 5,
 'Pitch_22': 6,
 'Pitch_23': 7,
 'Pitch_24': 8,
 'Pitch_25': 9,
 'Pitch_26': 10,
 'Pitch_27': 11,
 'Pitch_28': 12,
 'Pitch_29': 13,
 'Pitch_30': 14,
 'Pitch_31': 15,
 'Pitch_32': 16,
 'Pitch_33': 17,
 'Pitch_34': 18,
 'Pitch_35': 19,
 'Pitch_36': 20,
 'Pitch_37': 21,
 'Pitch_38': 22,
 'Pitch_39': 23,
 'Pitch_40': 24,
 'Pitch_41': 25,
 'Pitch_42': 26,
 'Pitch_43': 27,
 'Pitch_44': 28,
 'Pitch_45': 29,
 'Pitch_46': 30,
 'Pitch_47': 31,
 'Pitch_48': 32,
 'Pitch_49': 33,
 'Pitch_50': 34,
 'Pitch_51': 35,
 'Pitch_52': 36,
 'Pitch_53': 37,
 'Pitch_54': 38,
 'Pitch_55': 39,
 'Pitch_56': 40,
 'Pitch_57': 41,
 'Pitch_58': 42,
 'Pitch_59': 43,
 'Pitch_60': 44,
 'Pitch_61': 45,
 'Pitch_62': 46,
 'Pitch_63': 47,
 'Pitch_64': 48,
 'Pitch_65': 49,
 'Pitch_66': 50,
 'Pitch_67': 51,
 'Pitch_68': 52,
 'Pitch_69': 53,
 'Pitch_70': 54,
 'Pitch_71': 55,
 'Pitch_72': 56,
 'Pitch_73': 57,
 'Pitch_74': 58,
 'Pitc